# TEST FOR APP: API + PREDICTION

## SETUP

In [ ]:
# import libraries for paths
from pathlib import Path
import sys

# set paths
PATH_BASE = Path.cwd().parent.parent
if str(PATH_BASE) not in sys.path:
    sys.path.insert(0, str(PATH_BASE))

# import the functions from collector.py
import joblib # model import
import pandas as pd # data-wrangling
from src.jakob_analysis.functions import * # own functions
from src.bjarne_api.collector_new import Fetcher # own functions (bjarne)

# import models
pipe_mean = joblib.load(PATH_BASE / "src/jakob_analysis/pipeline_hgb_mean.pkl")
pipe_q05 = joblib.load(PATH_BASE / "src/jakob_analysis/pipeline_hgb_q05.pkl")
pipe_q95 = joblib.load(PATH_BASE / "src/jakob_analysis/pipeline_hgb_95.pkl")

# import lookup files
hist_delay_station_lookup = pd.read_parquet(PATH_BASE / "data/hist_delay_station_lookup.parquet")
hist_delay_train_lookup = pd.read_parquet(PATH_BASE / "data/hist_delay_train_lookup.parquet")
historical_features_list = [hist_delay_station_lookup, hist_delay_train_lookup]

## 1+2 SELECT START STATION + LOAD DATA FROM API

In [2]:
### DEFINE START STATION
start_station = "Berlin Hbf" # COMES FROM INPUT IN STREAMLIT (USER SHOULD CHOOSE)

### GET INFOS ON ALL RIDES
# Ausführung im Hauptprogramm 
if __name__ == "__main__":
    fetcher = Fetcher()
    station_name = start_station
    
    print(f"Suche Station ID für {station_name}...")
    station_id = fetcher.get_station_id(station_name)
    
    if station_id:
        print(f"Station ID gefunden: {station_id}. Suche Fernverkehrsabfahrten (nächste 60min)...")
        departures = fetcher.stations_details(station_id)
        
        if departures:
            print(f"{len(departures)} Abfahrten gefunden. Lade Details...")
            
            all_trips_dfs = []
            
            
            for i, departure in enumerate(departures, start=1):
                if "tripId" in departure:
                    trip_id = departure["tripId"]
                    line_name = departure.get("line", {}).get("name", "Unbekannt")
                    
                    print(f"Lade Trip {i}/{len(departures)}: {line_name}")
                    
                    trip_data = fetcher.trip_details(trip_id)
                    if trip_data is not None:
                        df_trip = fetcher.create_dataframe(trip_data, ride_id=i)
                    
                    if not df_trip.empty:
                        all_trips_dfs.append(df_trip)
                        
                    else: 
                        print(f"Details für Trip {line_name} konnten nicht verarbeitete werden")
            
            # Zusammenfügen aller DataFrames
            if all_trips_dfs:
                final_df = pd.concat(all_trips_dfs, ignore_index=True)
                print("\n--- FERTIG ---")
                print(final_df)
                print(f"\nGesamtanzahl Haltepunkte analysiert: {len(final_df)}")
                print(f"Anzahl Züge: {final_df['ride_id'].nunique()}")
            else:
                print("Konnte keine Trip-Details verarbeiten.")
        else:
            print("Keine passenden Abfahrten im Zeitraum gefunden.")
    else:
        print("Station nicht gefunden.")

# filter data
df_filtered = final_df[final_df["train_type"].isin(["ICE", "IC"])]

# get valid destinations (list format)
df_destinations = get_possible_destinations(df_filtered, start_station)

Suche Station ID für Berlin Hbf...
Station ID gefunden: 8011160. Suche Fernverkehrsabfahrten (nächste 60min)...
19 Abfahrten gefunden. Lade Details...
Lade Trip 1/19: ICE 2914
Lade Trip 2/19: ICE 509
Lade Trip 3/19: ICE 2919
Lade Trip 4/19: ICE 1908
Lade Trip 5/19: ICE 375
Lade Trip 6/19: BUS 40049
Lade Trip 7/19: ICE 1009
Lade Trip 8/19: ICE 2824
Lade Trip 9/19: FLX 1324
Lade Trip 10/19: ICE 548
Lade Trip 11/19: ICE 932
Lade Trip 12/19: ICE 931
Lade Trip 13/19: ICE 2831
Lade Trip 14/19: ICE 1050
Lade Trip 15/19: ICE 993
Lade Trip 16/19: BUS 40231
Lade Trip 17/19: ICE 691
Lade Trip 18/19: ICE 2863
Lade Trip 19/19: RJ 383

--- FERTIG ---
     ride_id train_name train_type       station_start     station_dest  \
0          1   ICE 2914        ICE  Frankfurt(Main)Hbf  Berlin Südkreuz   
1          1   ICE 2914        ICE  Frankfurt(Main)Hbf  Berlin Südkreuz   
2          1   ICE 2914        ICE  Frankfurt(Main)Hbf  Berlin Südkreuz   
3          1   ICE 2914        ICE  Frankfurt(Main)Hbf 

# 3 SELECT DESTINATION + GET CONNECTIONS

In [ ]:
# select destination
end_station = "Göttingen" # COMES FROM INPUT IN STREAMLIT (USER SHOULD CHOOSE)

# get rides between stations
df_connections = get_connections(df_filtered, start_station, end_station)
connections = df_connections

## 4 DISPLAY AND SELECT CONNECTION

In [ ]:
# work with df (for streamlit-harmonization)
df = connections

# get arrivals at end station for each train
arrivals = df[df["station_current"] == end_station].set_index("train_name")["arrival_planned"].to_dict()

df["print"] = df.apply(
    lambda x: (
        f"🚆 {x['train_name']} | "
        f"Dep: {pd.to_datetime(x['departure_planned']).strftime('%H:%M') if pd.notnull(x['departure_planned']) else '??:??'} → "
        f"Arr: {pd.to_datetime(arrivals.get(x['train_name'])).strftime('%H:%M') if pd.notnull(arrivals.get(x['train_name'])) else '??:??'} | "
        f"Delay: +{x['current_delay']} min"
    ), 
    axis=1)

# filter only relevant rows
filtered_options = df["print"][df["station_current"] == start_station].tolist()

# print out and select connection
print(filtered_options)
selected_connection = filtered_options[0] # COMES FROM INPUT IN STREAMLIT (USER SHOULD CHOOSE)

train_name = df[df["print"] == selected_connection]["train_name"].iloc[0]
df_selected = df[df["train_name"] == train_name]

['🚆 ICE 375 | Dep: 12:31 → Arr: 14:50 | Delay: +0 min', '🚆 ICE 993 | Dep: 13:20 → Arr: 15:50 | Delay: +0 min']


## 5 DATA WRANGLING FOR MODEL

In [13]:
# work with df (for streamlit-harmonization)
df = df_selected

# rename columns
df = df.rename({"precip": "precipitation", "temp": "temperature"}, axis=1)
# drop columns
df = df.drop(columns = ["current_delay", "stops_total", "stop_index", "stops_remaining"])

# create features
df_features = create_features(df, api = True, historical_features = historical_features_list)
df_final = df_features[df_features["station_current"] == start_station] # only one row

## 6 PREDICTION

In [15]:
def choose_features_target(df):

    cols_exclude = ["ride_id", # id
                    "delay", # target
                    "departure_real","arrival_real", "departure_planned", "arrival_planned", # raw time variables
                    "train_name", "station_current", "station_start", "station_dest", # high-dimensional categories 
                    "hist_delay_train_q90", "hist_delay_station_q90", "stops_total", "stop_index", "share_ride_time" # discarded after correlation analysis
                    ]
    
    feature_cols = [col for col in df.columns if col not in cols_exclude]
    return df[feature_cols]

X = choose_features_target(df_final)


pred_mean = pipe_mean.predict(X)[0]
pred_q05 = pipe_q05.predict(X)[0]
pred_q95 = pipe_q95.predict(X)[0]

NameError: name 'pipe_mean' is not defined

In [36]:

hgb_step = pipeline_hgb_mean.named_steps['histgradientboostingregressor']

# create dictionary 
model_params = {
    "learning_rate": hgb_step.learning_rate,
    "max_iter": hgb_step.max_iter,
    "max_leaf_nodes": hgb_step.max_leaf_nodes,
    "min_samples_leaf": hgb_step.min_samples_leaf,
}

print(model_params)

{'learning_rate': np.float64(0.010489929587681027), 'max_iter': 485, 'max_leaf_nodes': 34, 'min_samples_leaf': 79}
